# 28 Tagged model: choose the efficiency target (CPU runtime)

Scores the validation predictions of every target from `27_sft_tag_sweep`, then applies the method selection rule to the targets (the default +0.7 included): among the targets whose validation EX is not significantly below the best (paired McNemar test), the one with the most answers at least 2x cheaper than gold wins. Only the chosen target is then scored on dev and mini-dev.

**Runtime > Change runtime type > CPU** (timing runs on the same machine type as all earlier timing).

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
EXPORT = "sft_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
import json
KEPT = json.load(open(f"{R}/{EXPORT}_epoch_choice/selection.json"))["winner"]  # epoch 1 or 2, chosen in 25
assert json.load(open(f"{R}/{KEPT}/adapter_meta.json"))["arm"] == "tagged", KEPT
# Efficiency targets to try besides the default +0.7 (KEPT itself). Training tags: 73% +0.0, 2.7% >= +1.0, 1.1% at +3.0.
TAGS = ["+0.0", "+1.0", "+1.5", "+2.0", "+3.0"]
TAG_RUNS = [f"{KEPT}_tag{t}" for t in TAGS]
print("model:", KEPT)
print("target runs:", TAG_RUNS)
meta = json.load(open(f"{R}/{EXPORT}/export_meta.json"))
VAL_DBS = ",".join(meta["holdout_db_ids"])

### 1. Stage the validation databases and the dev databases

In [ ]:
!effsql stage --split train --db-ids {VAL_DBS}
!effsql stage --split dev

### 2. Score validation for every target (the +0.7 model was scored in `25`)

In [ ]:
for run in TAG_RUNS:
    !python -m effsql.train eval-val --run {run}

### 3. Choose the target on validation

In [ ]:
!python -m effsql.train select --runs {",".join([KEPT] + TAG_RUNS)} --out {EXPORT}_tag_choice
CHOSEN = json.load(open(f"{R}/{EXPORT}_tag_choice/selection.json"))["winner"]
print("chosen:", CHOSEN)

### 4. Dev and mini-dev for the chosen target (already done in `25` if it is the default +0.7)

In [ ]:
if CHOSEN != KEPT:
    !effsql evaluate --pred {R}/{CHOSEN}/predictions_dev.jsonl --split dev --timing none --run {CHOSEN}_dev --model {MODEL} --method sft-tagged-{CHOSEN.rsplit("_", 1)[1]}
    !effsql evaluate --pred {R}/{CHOSEN}/predictions_minidev.jsonl --split minidev --timing quick --run {CHOSEN}_minidev --model {MODEL} --method sft-tagged-{CHOSEN.rsplit("_", 1)[1]}
else:
    print("the default target +0.7 stays")

### 5. Results

In [ ]:
import pandas as pd
df = pd.read_csv(f"{DRIVE_ROOT}/effsql/results/results.csv")
df = df[df["run"].str.startswith((KEPT, "pool_v1_"))].drop_duplicates("run", keep="last")
cols = ["run", "split", "n", "ex", "rel_cost_median", "rel_cost_geomean", "eff_acc@1.5", "quick_ratio_median"]
print(df[cols].to_string(index=False))

### Done: disconnect the runtime